In [0]:
from pyspark.sql import SparkSession

data=[
        (1002,"malesh","thiru","thiru2@gmail.com","clerk"),
    (1003,"reddy","shray","reddy@gmail.com","supervisor")]

columns=["employeenumber","lastname","firstname","email","job"]

df= spark.createDataFrame(data, columns)
df.show()

#inspect Data
df.printSchema()

#cleaning Data





In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("Employee Pipeline").getOrCreate()
df =  spark.read.csv("/Volumes/de/thiru/t2/emp.csv",
    header=True,
    inferSchema= True,
    nullValue = "NULL"
)
df = df.toDF(*[c.strip() for c in df.columns])
print(df.columns)
df.show()

df_clean = df.withColumn("ename", upper(trim(col("ename"))))\
  .withColumn("job", lower(col("job")))\
  .withColumn("location", initcap(col("location"))) \
  .withColumn("ename",when(col("ename").isNull(),"unknown").otherwise(col("ename")))
  
df_clean.show() 

df_transformed = df_clean \
  .withColumn("full_name",concat_ws("-", col("ename"), col("job"))) \
  .withColumn("job_first_word", split(col("job"), " ")[0]) \
  .withColumn("name_length", length(col("ename"))) \
  .withColumn(
        "formatted_text",
        concat(lit("Name: "), col("ename"),
               lit(" | Job: "), col("job"),
               lit(" | Salary: "), col("salary"))
    )

df_transformed.show(truncate = False)

df_modified =df_transformed \
  .withColumn("job", regexp_replace(col("job"),"manager","lead")) \
  .withColumn("ename", regexp_replace(col("ename"),"A","Z"))\
  .withColumn("ename_reversed", reverse(col("ename"))) \
  .withColumn("ename_padded",lpad(col("ename"),10,"*"))

df_modified.show(truncate=False)

df_filtered = df_modified.filter(
    (col("ename").startswith("A")) | (col("job").contains("manager"))
)

df_filtered.show()

df_final_logic = df_modified.withColumn(
    "salary_category",
    when(col("salary") > 6000, "High")
    .when((col("salary") >= 4000) & (col("salary") <= 6000), "Medium")
    .otherwise("Low")
)
df_final_logic.show(truncate=False)

df_final=df_final_logic.select("emp_id","ename","job","salary","dept","location","salary","salary_category","formatted_text")
df_final.show()

df.groupby("dept").agg(sum("salary")).show()
df.groupby("dept").agg(avg("salary")).show()

df.createOrReplaceTempView("emp")

df.select(df.salary.isNull()).show()

df.select(
    when(col("salary").isNull(), lit("NA"))
    .when(col("salary") > 3000, col("salary"))
    .otherwise(col("salary"))
    .alias("salary_updated")
).show()


df.select(concat_ws('_',"ename","salary","dept")).show()
df.select(coalesce(col("ename"),lit("4"))).show()


In [0]:
# check existing files
!ls /Volumes/de/thiru/t2/

# create folders in allowed path
!mkdir -p /Volumes/de/thiru/t2/processed
!mkdir -p /Volumes/de/thiru/t2/archive

# verify
!ls /Volumes/de/thiru/t2/

In [0]:
# check file
!ls /Volumes/de/thiru/t2/

# create folder (if needed)
!mkdir -p /Volumes/de/thiru/t2/processed

# read file
df = spark.read.csv("/Volumes/de/thiru/t2/emp.csv", header=True, inferSchema=True)

df.show()

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark= SparkSession.builder.getOrCreate()

emp_data=[(1,"thiru",10),
          (2,"manu",20),
          (3,"nithin",30),
          (4,"hari",40)
          ]

dept_data=[(10,"hr"),
           (20,"IT"),
           (30,"finance")
           ]

salary_data=[(1,5000),
             (2,6000),  
             (3,7000),
             (5,6500)
            ]

emp_df=spark.createDataFrame(emp_data,["emp_id","emp_name","dept_id"])
dept_df=spark.createDataFrame(dept_data,["dept_id","dept_name"])
salary_df=spark.createDataFrame(salary_data,["emp_id","salary"])

# df = emp_df.join(dept_df, on="dept_id", how="inner")
# df= emp_df.join(dept_df, on="dept_id", how="left")

df= df.withColumn("dept_name",coalesce(col("dept_name"), lit("ty")))


df.show()

In [0]:
df= (
    emp_df 
   .join(dept_df,on="dept_id",how="left")
   .join(salary_df,on="emp_id", how="right")
)


df.createOrReplaceTempView("emp")

df.select(
    upper(col("emp_name")).alias("Upper_name"),
    lower(col("emp_name")).alias("lowe"),
    initcap(col("emp_name")).alias("initcap")
)
df.select(
    trim(col("emp_name")),
    ltrim(col("emp_name")),
    rtrim(col("emp_name"))
).show()

df.select(length(col("emp_name")).alias("length")).show()

df.select(concat_ws("-",col("emp_name"),col("dept_name")).alias("done")).show()

In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

path="/Volumes/de/thiru/t2/"

# customer_df = customer_df.toDF(*[c.strip() for c in customer_df.columns])
# print(customer_df.columns)

customers_df = spark.read.format("csv").option("header","true").option("inferSchema","true").option("nullValue","NULL") \
.load(f"{path}customers.csv")

employees_df= spark.read.format("csv").option("header","true").option("inferSchema","true").option("nullValue","NULL") \
 .load(f"{path}employees.csv")

offices_df= spark.read.format("csv").option("header","true").option("inferSchema","true").option("nullValue","NULL") \
.load(f"{path}offices.csv").show()
 
orderdetails_df=spark.read.format("csv").option("header","true").option("inferSchema","true").option("nullValue","Null") \
.load(f"{path}orderdetails.csv").show()

orders_df= spark.read.format("csv").option("header","true").option("inferSchema","true").option("nullValue","NULL") \
.load(f"{path}orders.csv").show()

payments_df=spark.read.format("csv").option("header","true").option("inferSchema","true").option("nullValue","Null") \
.load(f"{path}payments.csv").show()

productlines_df=spark.read.format("csv").option("header","true").option("inferSchema","true").option("nullValue","Null") \
.load(f"{path}productlines.csv").show()


products_df=spark.read.format("csv").option("header","true").option("inferSchema","true").option("nullValue","Null") \
.load(f"{path}products.csv").show()

print(type(employees_df))

# customers_df.select(count("*").alias("total"),
#                    *[sum(col(c).isNull().cast("int")).alias(f"{c}_nullcount")for c in customers_df.columns],
#                    *[count(col(c)).alias(f"{c}_notnull")for c in customers_df.columns]
#                    ).display()

cust_emp_df = customers_df.join(
    employees_df,
    customers_df.salesRepEmployeeNumber == employees_df.employeeNumber,
    "inner"
)

cust_emp_df.select(
    "customerName",
    "employeeNumber",
    concat(col("firstName"), lit(" "), col("lastName")).alias("full_name")
).display()





In [0]:

# df = spark.read \
#     .option("header", "true") \
#     .option("inferSchema", "true") \
#     .csv("/Volumes/de/thiru/t2/emp.csv")

# df.write \
#     .mode("overwrite") \
#     .parquet("/Volumes/de/thiru/t2/emp_parquet")

# df_parquet = spark.read.parquet("/Volumes/de/thiru/t2/emp_parquet")
# df_parquet.show()

